In [ ]:
# clone repository
!git clone https://github.com/zazuay/Generative-Models-Benchmark.git
%cd Generative-Models-Benchmark/
!pip install -q -r requirements.txt

fatal: destination path 'Generative-Models-Benchmark' already exists and is not an empty directory.
/content/Generative-Models-Benchmark


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# import libraries
import os
import json
import torch
import time
import random
import numpy as np
import torch.nn as nn
import copy
import math
import argparse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import ConcatDataset, DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid

print("success")

success


In [ ]:
# load dataset
import kagglehub

path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files: ", path)

SEED = 42

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files:  /kaggle/input/fashionmnist


In [ ]:
# utils
def get_device():
  return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def count_params(m):
  return sum(p.numel() for p in m.parameters())

def save_json(obj, path):
  os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
  with open(path, "w") as f:
    json.dump(obj, f, indent=2)

def load_json(path):
  with open(path) as f:
    return json.load(f)

In [ ]:
# sanity check data
def set_seed(seed: int = SEED):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

def get_datasets(root: str= "./data"):
  if os.environ.get("DATA_PATH") == "1":
    g = torch.Generator().manual_seed()
    full = TensorDataset(
        torch.rand(1000, 1, 28, 28, generator=g),
        torch.randint(0, 10, (1000,), generator=g)
    )
  else:
    tf = transforms.ToTensor()
    train_dataset = datasets.FashionMNIST(root=root, train=True, download=True, transform=tf)
    test_dataset = datasets.FashionMNIST(root=root, train=False, download=True, transform=tf)
    full = ConcatDataset([train_dataset, test_dataset])

  n = len(full)
  n_train = int(0.8 * n)
  n_val = int(0.1 * n)
  n_test = n - n_train - n_val

  gen = torch.Generator().manual_seed(SEED)

  return random_split(full, [n_train, n_val, n_test], generator=gen)

def get_loaders(batch_size: int = 128, root: str = "./data", num_workers: int = 2):
  train, val, test = get_datasets(root)
  pin = torch.cuda.is_available()
  g = torch.Generator().manual_seed(SEED)
  kw = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=pin)
  return {
      "train" : DataLoader(train, shuffle=True, generator=g, **kw),
      "val" : DataLoader(val, shuffle=False, **kw),
      "test" : DataLoader(test, shuffle=False, **kw)
  }

set_seed()
ld = get_loaders()
print({k: len(v.dataset) for k, v in ld.items()})
x, y = next(iter(ld["train"]))
print(x.shape, x.min().item(), x.max().item(), y[:10])

{'train': 56000, 'val': 7000, 'test': 7000}
torch.Size([128, 1, 28, 28]) 0.0 1.0 tensor([7, 4, 1, 7, 4, 2, 3, 3, 9, 3])


In [ ]:
# metrics
def mse_per_image(x, y):
  return((x - y) ** 2).flatten(1).mean(1)

def psnr_per_image(x, y, max_val: float=1.0):
  return 10 * torch.log10(max_val ** 2 / mse_per_image(x, y).clamp_min(1e-10))

def _gauss_kernel(win: int=11, sigma: float=1.5, device="cuda"):
  c = torch.arange(win, dtype=torch.float32, device=device) - win // 2
  g = torch.exp(-(c ** 2) / (2 * sigma ** 2))
  g = g / g.sum()
  return (g[:, None] * g[None, :])[None, None]

def ssim_per_image(x, y, max_val: float = 1.0):
  C1, C2 = (0.01 * max_val) ** 2, (0.03 * max_val) ** 2
  w = _gauss_kernel(device=x.device)
  mu_x, mu_y = F.conv2d(x, w), F.conv2d(y, w)
  sxx = F.conv2d(x * x, w) - mu_x ** 2
  syy = F.conv2d(y * y, w) - mu_y ** 2
  sxy = F.conv2d(x * y, w) - mu_x * mu_y
  s = ((2 * mu_x * mu_y + C1) * (2 * sxy + C2)) / ((mu_x ** 2 + mu_y ** 2 + C1) * (sxx + syy + C2))
  return s.flatten(1).mean(1)

@torch.no_grad()
def recon_metrics(x, x_hat, bs: int = 1000):
  m, p, s, n = 0.0, 0.0, 0.0, x.size(0)
  for i in range(0, n, bs):
    a, b = x[i:i + bs], x_hat[i:i + bs]
    m += mse_per_image(a, b).sum().item()
    p += psnr_per_image(a, b).sum().item()
    s += ssim_per_image(a, b).sum().item()
  return {"mse": m / n, "psnr": p / n, "ssim": s / n}

def sync(dev):
  if dev.type == "cuda":
    torch.cuda.synchronize()

@torch.no_grad()
def measure_latency(fn, dev, warmup: int =  5, iters: int = 20):
  for _ in range(warmup):
    fn()
  sync(dev)
  t0 = time.perf_counter()
  for _ in range(iters):
    fn()
  sync(dev)
  return (time.perf_counter() - t0) / iters * 1000

In [ ]:
# model AE
class AE(nn.Module):
  def __init__(self, latent=64, hidden=256, sparse=False):
    super().__init__()
    enc = [nn.Flatten(), nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, latent)]
    if sparse:
      enc.append(nn.ReLU())
    self.encoder = nn.Sequential(*enc)
    self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(),
                                   nn.Linear(hidden, 784), nn.Sigmoid())
    self.latent = latent

  def encode(self, x): return self.encoder(x)

  def decode(self, z): return self.decoder(z).view(-1, 1, 28, 28)

  def forward(self, x):
    z = self.encode(x)
    return self.decode(z), z

  def encode_mean(self, x): return self.encode(x)

  def reconstruct(self, x): return self(x)[0]

In [ ]:
# model VAE
class VAE(nn.Module):
  def __init__(self, latent=64, hidden=256):
    super().__init__()
    self.enc = nn.Sequential(nn.Flatten(), nn.Linear(784, hidden), nn.ReLU())
    self.fc_mu = nn.Linear(hidden, latent)
    self.fc_logvar = nn.Linear(hidden, latent)
    self.dec = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, 784))
    self.latent = latent

  def encode(self, x):
    h = self.enc(x)
    return self.fc_mu(h), self.fc_logvar(h)

  def decode_logits(self, z): return self.dec(z).view(-1, 1, 28, 28)

  def decode(self, z): return torch.sigmoid(self.decode_logits(z))

  def forward(self, x):
    mu, logvar = self.encode(x)
    z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
    return self.decode_logits(z), mu, logvar

  def encode_mean(self, x): return self.encode(x)[0]

  def reconstruct(self, x): return self.decode(self.encode(x)[0])

In [ ]:
# model GAN
def weights_init(m):
  name = m.__class__.__name__
  if "Conv" in name or "Linear" in name:
    nn.init.normal_(m.weight, 0.0, 0.02)
    if m.bias is not None:
      nn.init.zeros_(m.bias)
  elif "BatchNorm" in name:
    nn.init.normal_(m.weight, 1.0, 0.02)
    nn.init.zeros_(m.bias)

class MLPGenerator(nn.Module):
  def __init__(self, zdim=64):
    super().__init__()
    self.zdim = zdim
    self.net = nn.Sequential(
        nn.Linear(zdim, 256), nn.LeakyReLU(0.2),
        nn.Linear(256, 512), nn.LeakyReLU(0.2),
        nn.Linear(512, 1024), nn.LeakyReLU(0.2),
        nn.Linear(1024, 784), nn.Tanh()
    )
  def forward(self, z):
    return self.net(z).view(-1, 1, 28, 28)

class MLPDiscriminator(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 1024), nn.LeakyReLU(0.2), nn.Dropout(0.3),
        nn.Linear(1024, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
        nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
        nn.Linear(256, 1)
    )
  def forward(self, x):
    return self.net(x).squeeze(1)

class DCGenerator(nn.Module):
  def __init__(self, zdim=100):
    super().__init__()
    self.zdim = zdim
    self.net = nn.Sequential(
        nn.ConvTranspose2d(zdim, 128, 7, 1, 0, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
        nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(True),
        nn.ConvTranspose2d(64, 1, 4, 2, 1), nn.Tanh()
    )
  def forward(self, z):
    return self.net(z.view(z.size(0), self.zdim, 1, 1))

class DCDiscriminator(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2, True),
        nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
        nn.Conv2d(128, 1, 7, 1, 0)
    )
  def forward(self, x):
    return self.net(x).view(-1)

In [ ]:
# model DDPM
class SinEmb(nn.Module):
  def __init__(self, dim):
    super().__init__()
    self.dim = dim
  def forward(self, t):
    half = self.dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    a = t.float()[:, None] * freqs[None]
    return torch.cat([a.sin(), a.cos()], dim=1)

class ResBlock(nn.Module):
  def __init__(self, cin, cout, tdim):
    super().__init__()
    self.n1 = nn.GroupNorm(8, cin)
    self.c1 = nn.Conv2d(cin, cout, 3, padding=1)
    self.t = nn.Linear(tdim, cout)
    self.n2 = nn.GroupNorm(8, cout)
    self.c2 = nn.Conv2d(cout, cout, 3, padding=1)
    self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
  def forward(self, x, t):
    h = self.c1(F.silu(self.n1(x)))
    h = h + self.t(F.silu(t))[:, :, None, None]
    h = self.c2(F.silu(self.n2(h)))
    return h + self.skip(x)

class UNet(nn.Module):
  def __init__(self, base=64):
    super().__init__()
    b, td = base, base * 4
    self.temb = nn.Sequential(SinEmb(b), nn.Linear(b, td), nn.SiLU(), nn.Linear(td, td))
    self.inc = nn.Conv2d(1, b, 3, padding=1)
    self.r1 = ResBlock(b, b, td)
    self.d1 = nn.Conv2d(b, b, 3, 2, 1)
    self.r2 = ResBlock(b, 2 * b, td)
    self.d2 = nn.Conv2d(2 * b, 2 * b, 3, 2, 1)
    self.r3 = ResBlock(2 * b, 2 * b, td)
    self.mid = ResBlock(2 * b, 2 * b, td)
    self.u3 = ResBlock(4 * b, 2 * b, td)
    self.up1 = nn.Conv2d(2 * b, 2 * b, 3, padding=1)
    self.u2 = ResBlock(4 * b, b, td)
    self.up2 = nn.Conv2d(b, b, 3, padding=1)
    self.u1 = ResBlock(2 * b, b, td)
    self.out = nn.Sequential(nn.GroupNorm(8, b), nn.SiLU(), nn.Conv2d(b, 1, 3, padding=1))
  def forward(self, x, t):
    t = self.temb(t)
    r1 = self.r1(self.inc(x), t)
    r2 = self.r2(self.d1(r1), t)
    r3 = self.r3(self.d2(r2), t)
    m = self.mid(r3, t)
    u3 = self.u3(torch.cat([m, r3], 1), t)
    up1 = self.up1(F.interpolate(u3, scale_factor=2))
    u2 = self.u2(torch.cat([up1, r2], 1), t)
    up2 = self.up2(F.interpolate(u2, scale_factor=2))
    u1 = self.u1(torch.cat([up2, r1], 1), t)
    return self.out(u1)

class Diffusion:
  def __init__(self, T=1000, beta1=1e-4, betaT=0.02, device="cuda"):
    self.T = T
    self.betas = torch.linspace(beta1, betaT, T, device=device)
    self.alphas = 1.0 - self.betas
    self.abar = torch.cumprod(self.alphas, 0)

  def to(self, device):
    for k in ("betas", "alphas", "abar"):
      setattr(self, k, getattr(self, k).to(device))
    return self

  def q_sample(self, x0, t, noise):
    a = self.abar[t][:, None, None, None]
    return a.sqrt() * x0 + (1 - a).sqrt() * noise

  def loss(self, model, x0, gen=None):
    B = x0.size(0)
    t = torch.randint(0, self.T, (B,), device=x0.device, generator=gen)
    noise = torch.randn(x0.shape, device=x0.device, generator=gen)
    return F.mse_loss(model(self.q_sample(x0, t, noise), t), noise)

  @torch.no_grad()
  def sample(self, model, n, device, save_ts=None, gen=None):
    x = torch.randn(n, 1, 28, 28, device=device, generator=gen)
    traj = []
    for t in reversed(range(self.T)):
      if save_ts is not None and t in save_ts:
        traj.append((t, x.clone()))
      tt = torch.full((n,), t, device=device, dtype=torch.long)
      eps = model(x, tt)
      a, ab, b = self.alphas[t], self.abar[t], self.betas[t]
      mean = (x - b / (1 - ab).sqrt() * eps) / a.sqrt()
      x = mean + b.sqrt() * torch.randn(x.shape, device=device, generator=gen) if t > 0 else mean
    x = x.clamp(-1, 1)
    if save_ts is not None:
      traj.append((-1, x.clone()))
      return x, traj
    return x

class EMA:
  def __init__(self, model, decay=0.999):
    self.module = copy.deepcopy(model).eval()
    for p in self.module.parameters():
      p.requires_grad_(False)
    self.decay = decay

  @torch.no_grad()
  def update(self, model):
    for pe, p in zip(self.module.parameters(), model.parameters()):
      pe.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)

In [ ]:
DEFAULTS = {
    "ae": dict(latent=64, hidden=256),
    "sparse": dict(latent=256, hidden=512, lam=1e-3),
    "vae": dict(latent=64, hidden=256, beta=1.0),
    "gan": dict(zdim=64),
    "dcgan": dict(zdim=100),
    "ddpm": dict(base=64, T=1000),
}
LR = {"ae": 1e-3, "sparse": 1e-3, "vae": 1e-3, "gan": 2e-4, "dcgan": 2e-4, "ddpm": 2e-4}
ALL = list(DEFAULTS)

In [ ]:
# build model
def build_model(name, cfg):
  if name == "ae":
    return AE(cfg["latent"], cfg["hidden"], sparse=False)
  if name == "sparse":
    return AE(cfg["latent"], cfg["hidden"], sparse=True)
  if name == "vae":
    return VAE(cfg["latent"], cfg["hidden"])
  if name == "gan":
    return MLPGenerator(cfg["zdim"])
  if name == "dcgan":
    return DCGenerator(cfg["zdim"])
  if name == "ddpm":
    return UNet(cfg["base"])
  raise ValueError(name)

def ckpt_path(runs, name):
  for f in ("best.pt", "last.pt"):
    p = os.path.join(runs, name, f)
    if os.path.exists(p):
      return p
  return None

def load_model(name, runs, device):
  p = ckpt_path(runs, name)
  if p is None:
    return None
  ck = torch.load(p, map_location=device, weights_only=False)
  cfg = ck["cfg"]
  model = build_model(name, cfg).to(device)
  key = {"gan": "G", "dcgan": "G", "ddpm": "ema"}.get(name, "model")
  model.load_state_dict(ck[key])
  model.eval()
  diff = Diffusion(cfg["T"], device=device) if name == "ddpm" else None
  return model, cfg, diff, ck

@torch.no_grad()
def generate(name, model, cfg, diff, n, device, bs=500, gen=None):
  out = []
  for i in range(0, n, bs):
    m = min(bs, n - i)
    if name == "vae":
      x = model.decode(torch.randn(m, cfg["latent"], device=device, generator=gen))
    elif name in ("gan", "dcgan"):
      x = model(torch.randn(m, cfg["zdim"], device=device, generator=gen)) * 0.5 + 0.5
    elif name == "ddpm":
      x = diff.sample(model, m, device, gen=gen)
    else:
      raise ValueError(f"{name} tidak punya sampling dari prior")
    out.append(x.cpu())
  return torch.cat(out)

In [ ]:
# model train
def parse(argv=None):
  p = argparse.ArgumentParser()
  p.add_argument("--model", required=True, choices=list(DEFAULTS))
  p.add_argument("--epochs", type=int, default=30)
  p.add_argument("--lr", type=float, default=None)
  p.add_argument("--batch_size", type=int, default=128)
  p.add_argument("--runs", default="runs")
  p.add_argument("--data_root", default="./data")
  p.add_argument("--num_workers", type=int, default=2)
  p.add_argument("--resume", action="store_true")
  p.add_argument("--sample_every", type=int, default=5, help="simpan grid sampel tiap N epoch")
  p.add_argument("--max_batches", type=int, default=None, help="batasi batch/epoch (smoke test)")
  # override konfigurasi
  p.add_argument("--latent", type=int)
  p.add_argument("--hidden", type=int)
  p.add_argument("--lam", type=float, help="bobot penalti L1 Sparse AE")
  p.add_argument("--beta", type=float, help="bobot KL VAE")
  p.add_argument("--zdim", type=int)
  p.add_argument("--base", type=int, help="lebar UNet DDPM")
  p.add_argument("--T", type=int, help="jumlah timestep difusi")
  return p.parse_args(argv)

def limited(loader, n):
  for i, b in enumerate(loader):
    if n is not None and i >= n:
      break
    yield b

def load_resume(args, out):
  path = os.path.join(out, "last.pt")
  if args.resume and os.path.exists(path):
    print(f"[resume] {path}")
    return torch.load(path, map_location="cuda", weights_only=False)
  return None

In [ ]:
# train AE
def train_ae_like(args, cfg, loaders, dev, out):
  name = args.model
  model = build_model(name, cfg).to(dev)
  opt = torch.optim.Adam(model.parameters(), lr=args.lr)
  log, best, start = [], float("inf"), 0
  ck = load_resume(args, out)
  if ck:
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    log, best, start = ck["log"], ck["best"], ck["epoch"]

  def loss_fn(x):
    B = x.size(0)
    if name == "vae":
      logits, mu, logvar = model(x)
      rec = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / B
      kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / B
      loss = rec + cfg["beta"] * kl
      recon, extra = torch.sigmoid(logits), {"recon_loss": rec.item(), "kl": kl.item()}
    else:
      recon, z = model(x)
      mse = F.mse_loss(recon, x)
      if name == "sparse":
        l1 = z.abs().mean()
        loss, extra = mse + cfg["lam"] * l1, {"l1": l1.item()}
      else:
        loss, extra = mse, {}
    extra["pix_mse"] = F.mse_loss(recon, x).item()
    return loss, extra

  def run(loader, train):
    model.train(train)
    tot, n, acc = 0.0, 0, {}
    with torch.set_grad_enabled(train):
      for x, _ in limited(loader, args.max_batches):
        x = x.to(dev)
        loss, extra = loss_fn(x)
        if train:
          opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        B = x.size(0); tot += loss.item() * B; n += B
        for k, v in extra.items():
          acc[k] = acc.get(k, 0.0) + v * B
    return tot / n, {k: v / n for k, v in acc.items()}

  for ep in range(start + 1, args.epochs + 1):
    sync(dev); t0 = time.perf_counter()
    tr, tr_x = run(loaders["train"], True)
    sync(dev); et = time.perf_counter() - t0
    va, va_x = run(loaders["val"], False)
    row = dict(epoch=ep, epoch_time=et, train_loss=tr, val_loss=va,
               train_mse=tr_x["pix_mse"], val_mse=va_x["pix_mse"],
               **{f"val_{k}": v for k, v in va_x.items() if k != "pix_mse"})
    log.append(row)
    print({k: round(v, 5) if isinstance(v, float) else v for k, v in row.items()})
    state = dict(model=model.state_dict(), opt=opt.state_dict(), cfg=cfg, log=log,
                 best=min(best, va), epoch=ep)
    if va < best:
      best = va; torch.save(state, os.path.join(out, "best.pt"))
    torch.save(state, os.path.join(out, "last.pt"))
    save_json(log, os.path.join(out, "log.json"))

In [ ]:
# train GAN
def train_gan(args, cfg, loaders, dev, out):
  name = args.model
  G = build_model(name, cfg).to(dev)
  D = (MLPDiscriminator() if name == "gan" else DCDiscriminator()).to(dev)
  if name == "dcgan":
    G.apply(weights_init); D.apply(weights_init)
  oG = torch.optim.Adam(G.parameters(), lr=args.lr, betas=(0.5, 0.999))
  oD = torch.optim.Adam(D.parameters(), lr=args.lr, betas=(0.5, 0.999))
  log, start = [], 0
  ck = load_resume(args, out)
  if ck:
    G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
    oG.load_state_dict(ck["oG"]); oD.load_state_dict(ck["oD"]); log, start = ck["log"], ck["epoch"]
  fixed_z = torch.randn(64, cfg["zdim"], generator=torch.Generator().manual_seed(0)).to(dev)
  bce = F.binary_cross_entropy_with_logits

  for ep in range(start + 1, args.epochs + 1):
    G.train(); D.train()
    sync(dev); t0 = time.perf_counter()
    s = dict(d=0.0, g=0.0, dr=0.0, df=0.0); n = 0
    for x, _ in limited(loaders["train"], args.max_batches):
      x = x.to(dev) * 2 - 1
      B = x.size(0)
      fake = G(torch.randn(B, cfg["zdim"], device=dev))

      lr_, lf_ = D(x), D(fake.detach())
      lossD = bce(lr_, torch.ones_like(lr_)) + bce(lf_, torch.zeros_like(lf_))
      oD.zero_grad(set_to_none=True); lossD.backward(); oD.step()

      lg = D(fake)
      lossG = bce(lg, torch.ones_like(lg))
      oG.zero_grad(set_to_none=True); lossG.backward(); oG.step()
      s["d"] += lossD.item() * B; s["g"] += lossG.item() * B
      s["dr"] += torch.sigmoid(lr_).mean().item() * B; s["df"] += torch.sigmoid(lf_).mean().item() * B
      n += B
    sync(dev); et = time.perf_counter() - t0
    row = dict(epoch=ep, epoch_time=et, loss_d=s["d"] / n, loss_g=s["g"] / n,
               d_real=s["dr"] / n, d_fake=s["df"] / n)
    log.append(row)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})
    if ep % args.sample_every == 0 or ep == args.epochs:
      G.eval()
      with torch.no_grad():
        save_image(G(fixed_z) * 0.5 + 0.5, os.path.join(out, "samples", f"epoch_{ep:03d}.png"), nrow=8)
    torch.save(dict(G=G.state_dict(), D=D.state_dict(), oG=oG.state_dict(), oD=oD.state_dict(),
                    cfg=cfg, log=log, epoch=ep), os.path.join(out, "last.pt"))
    save_json(log, os.path.join(out, "log.json"))


In [ ]:
# train DDPM
def train_ddpm(args, cfg, loaders, dev, out):
  model = build_model("ddpm", cfg).to(dev)
  ema = EMA(model, 0.999)
  diff = Diffusion(cfg["T"], device=dev)
  opt = torch.optim.Adam(model.parameters(), lr=args.lr)
  log, best, start = [], float("inf"), 0
  ck = load_resume(args, out)
  if ck:
    model.load_state_dict(ck["model"]); ema.module.load_state_dict(ck["ema"])
    opt.load_state_dict(ck["opt"]); log, best, start = ck["log"], ck["best"], ck["epoch"]
  vgen = torch.Generator(device=dev)

  for ep in range(start + 1, args.epochs + 1):
    model.train()
    sync(dev); t0 = time.perf_counter()
    tot, n = 0.0, 0
    for x, _ in limited(loaders["train"], args.max_batches):
      x = x.to(dev) * 2 - 1
      loss = diff.loss(model, x)
      opt.zero_grad(set_to_none=True); loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
      opt.step(); ema.update(model)
      tot += loss.item() * x.size(0); n += x.size(0)
    sync(dev); et = time.perf_counter() - t0
    model.eval(); vt, vn = 0.0, 0
    vgen.manual_seed(0)
    with torch.no_grad():
      for x, _ in limited(loaders["val"], args.max_batches):
        x = x.to(dev) * 2 - 1
        vt += diff.loss(model, x, gen=vgen).item() * x.size(0); vn += x.size(0)
    va = vt / vn
    row = dict(epoch=ep, epoch_time=et, train_loss=tot / n, val_loss=va)
    log.append(row)
    print({k: round(v, 5) if isinstance(v, float) else v for k, v in row.items()})
    state = dict(model=model.state_dict(), ema=ema.module.state_dict(), opt=opt.state_dict(),
                 cfg=cfg, log=log, best=min(best, va), epoch=ep)
    if va < best:
      best = va; torch.save(state, os.path.join(out, "best.pt"))
    torch.save(state, os.path.join(out, "last.pt"))
    save_json(log, os.path.join(out, "log.json"))
    if ep % args.sample_every == 0 or ep == args.epochs:
      x = diff.sample(ema.module, 16, dev) * 0.5 + 0.5
      save_image(x, os.path.join(out, "samples", f"epoch_{ep:03d}.png"), nrow=4)

In [ ]:
def main_train(argv=None):
  args = parse(argv)
  set_seed()
  dev = get_device()
  cfg = dict(DEFAULTS[args.model])
  for k in cfg:
    if getattr(args, k, None) is not None:
      cfg[k] = getattr(args, k)
  args.lr = args.lr or LR[args.model]
  out = os.path.join(args.runs, args.model)
  os.makedirs(os.path.join(out, "samples"), exist_ok=True)
  print(f"device={dev} model={args.model} cfg={cfg} lr={args.lr} epochs={args.epochs}")
  loaders = get_loaders(args.batch_size, args.data_root, args.num_workers)
  fn = {"ae": train_ae_like, "sparse": train_ae_like, "vae": train_ae_like,
        "gan": train_gan, "dcgan": train_gan, "ddpm": train_ddpm}[args.model]
  fn(args, cfg, loaders, dev, out)

In [ ]:
RUNS = "/content/drive/MyDrive/AssignmentDLdrive"

In [ ]:
main_train(["--model", "ae", "--epochs", "30", "--runs", RUNS, "--resume"])

device=cuda model=ae cfg={'latent': 64, 'hidden': 256} lr=0.001 epochs=30
[resume] /content/drive/MyDrive/AssignmentDLdrive/ae/last.pt


In [ ]:
main_train(["--model", "vae", "--epochs", "30", "--runs", RUNS, "--resume"])

device=cuda model=vae cfg={'latent': 64, 'hidden': 256, 'beta': 1.0} lr=0.001 epochs=30
[resume] /content/drive/MyDrive/AssignmentDLdrive/vae/last.pt


In [ ]:
main_train(["--model", "gan", "--epochs", "50", "--runs", RUNS, "--resume"])

device=cuda model=gan cfg={'zdim': 64} lr=0.0002 epochs=50
[resume] /content/drive/MyDrive/AssignmentDLdrive/gan/last.pt


In [ ]:
main_train(["--model", "dcgan", "--epochs", "30", "--runs", RUNS, "--resume"])

device=cuda model=dcgan cfg={'zdim': 100} lr=0.0002 epochs=30
[resume] /content/drive/MyDrive/AssignmentDLdrive/dcgan/last.pt


In [ ]:
main_train(["--model", "ddpm", "--epochs", "30", "--runs", RUNS, "--resume", "--sample_every", "10"])

device=cuda model=ddpm cfg={'base': 64, 'T': 1000} lr=0.0002 epochs=30
[resume] /content/drive/MyDrive/AssignmentDLdrive/ddpm/last.pt


In [ ]:
import shutil
for lam in ["1e-4", "1e-3", "1e-2"]:
  shutil.rmtree(f"{RUNS}/sweep_lam{lam}", ignore_errors=True)
shutil.rmtree(f"{RUNS}/sparse", ignore_errors=True)
for lam in ["1e-4", "1e-3", "1e-2"]:
  main_train(["--model", "sparse", "--epochs", "30", "--lam", lam,
              "--runs", f"{RUNS}/sweep_lam{lam}"])

device=cuda model=sparse cfg={'latent': 256, 'hidden': 512, 'lam': 0.0001} lr=0.001 epochs=30
{'epoch': 1, 'epoch_time': 8.48818, 'train_loss': 0.03059, 'val_loss': 0.01834, 'train_mse': 0.0305, 'val_mse': 0.01824, 'val_l1': 0.92865}
{'epoch': 2, 'epoch_time': 7.58691, 'train_loss': 0.01582, 'val_loss': 0.0147, 'train_mse': 0.01573, 'val_mse': 0.01461, 'val_l1': 0.89796}
{'epoch': 3, 'epoch_time': 12.50857, 'train_loss': 0.01273, 'val_loss': 0.01192, 'train_mse': 0.01264, 'val_mse': 0.01183, 'val_l1': 0.94533}
{'epoch': 4, 'epoch_time': 8.4625, 'train_loss': 0.01116, 'val_loss': 0.01091, 'train_mse': 0.01107, 'val_mse': 0.01082, 'val_l1': 0.93435}
{'epoch': 5, 'epoch_time': 8.76036, 'train_loss': 0.01009, 'val_loss': 0.00996, 'train_mse': 0.01, 'val_mse': 0.00987, 'val_l1': 0.90727}
{'epoch': 6, 'epoch_time': 6.81912, 'train_loss': 0.00924, 'val_loss': 0.00908, 'train_mse': 0.00915, 'val_mse': 0.00899, 'val_l1': 0.92651}
{'epoch': 7, 'epoch_time': 8.08136, 'train_loss': 0.00864, 'val_l

In [ ]:
# Bandingkan 3 lambda: MSE vs sparsity SEBENARNYA (fraksi unit nol), bukan hanya mean |z|
RUNS = "/content/drive/MyDrive/AssignmentDLdrive"
dev = get_device()
xs = torch.cat([x for x, _ in get_loaders(128, "./data", 2)["test"]])
print(f"{'lam':>6} | {'test MSE':>9} | {'mean|z|':>8} | {'frac z==0':>9} | {'dead units':>10}")
for lam in ["1e-4", "1e-3", "1e-2"]:
    m = load_model("sparse", f"{RUNS}/sweep_lam{lam}", dev)[0]
    with torch.no_grad():
        z = torch.cat([m.encode(xs[i:i + 1000].to(dev)).cpu() for i in range(0, len(xs), 1000)])
        rec = torch.cat([m.reconstruct(xs[i:i + 1000].to(dev)).cpu() for i in range(0, len(xs), 1000)])
    print(f"{lam:>6} | {mse_per_image(xs, rec).mean().item():9.5f} | {z.abs().mean().item():8.4f} | "
          f"{(z == 0).float().mean().item():9.3f} | {int((z == 0).all(0).sum().item()):10d}")


   lam |  test MSE |  mean|z| | frac z==0 | dead units
  1e-4 |   0.00443 |   0.8301 |     0.292 |         63
  1e-3 |   0.00444 |   0.3293 |     0.377 |         66
  1e-2 |   0.00550 |   0.0692 |     0.630 |        122


In [ ]:
# Pilih lambda SETELAH melihat tabel di atas (default 1e-3 hanya tebakan awal), lalu salin.
import shutil
BEST_LAM = "1e-3"
shutil.rmtree(f"{RUNS}/sparse", ignore_errors=True)
shutil.copytree(f"{RUNS}/sweep_lam{BEST_LAM}/sparse", f"{RUNS}/sparse")
print("sparse <-", BEST_LAM)


sparse <- 1e-3


In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# MLP experiment
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1, self.fc2, self.fc3 = nn.Linear(784, 256), nn.Linear(256, 64), nn.Linear(64, 10)

    def features(self, x):
        h1 = F.relu(self.fc1(x.flatten(1)))
        return h1, F.relu(self.fc2(h1))          # z_MLP = aktivasi ReLU layer 64-d

    def forward(self, x):
        return self.fc3(self.features(x)[1])


def collect(loader, dev):
    xs, ys = zip(*[(x, y) for x, y in loader])
    return torch.cat(xs), torch.cat(ys)


@torch.no_grad()
def accuracy(model, x, y, dev, bs=1000):
    c = sum((model(x[i:i + bs].to(dev)).argmax(1).cpu() == y[i:i + bs]).sum().item() for i in range(0, len(x), bs))
    return c / len(x)


def orth_stats(W):
    """W: (out, in). Untuk out<in, W^T W (in x in) berpangkat <= out sehingga PASTI != I (bukti trivial);
    W W^T (out x out) lebih informatif: seberapa ortonormal baris-barisnya."""
    out_d, in_d = W.shape
    WtW, WWt = W.T @ W, W @ W.T
    sv = torch.linalg.svdvals(W)
    return {"shape": [out_d, in_d],
            "fro_WtW_minus_I": (WtW - torch.eye(in_d)).norm().item(),
            "fro_WWt_minus_I": (WWt - torch.eye(out_d)).norm().item(),
            "fro_WWt_minus_I_normalized": ((WWt - torch.eye(out_d)).norm() / out_d ** 0.5).item(),
            "sv_max": sv.max().item(), "sv_min": sv.min().item(), "cond_number": (sv.max() / sv.min()).item()}

In [ ]:
def main_mlp(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", default="runs")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--dec_epochs", type=int, default=30)
    ap.add_argument("--data_root", default="./data")
    ap.add_argument("--num_workers", type=int, default=2)
    args = ap.parse_args(argv)
    set_seed()
    dev = get_device()
    out = os.path.join(args.runs, "mlp"); os.makedirs(out, exist_ok=True)
    L = get_loaders(128, args.data_root, args.num_workers)
    xtr, ytr = collect(L["train"], dev); xva, yva = collect(L["val"], dev); xte, yte = collect(L["test"], dev)
    R = {}

    # 1. latih MLP (CE)
    mlp = MLP().to(dev); opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
    best, best_sd = 0.0, None
    for ep in range(1, args.epochs + 1):
        mlp.train()
        for x, y in L["train"]:
            loss = F.cross_entropy(mlp(x.to(dev)), y.to(dev))
            opt.zero_grad(); loss.backward(); opt.step()
        mlp.eval(); va = accuracy(mlp, xva, yva, dev)
        if va > best:
            best, best_sd = va, {k: v.clone() for k, v in mlp.state_dict().items()}
        print(f"[T1] epoch {ep} val_acc={va:.4f}")
    mlp.load_state_dict(best_sd); mlp.eval()
    for p in mlp.parameters():
        p.requires_grad_(False)                       # dibekukan untuk Tahap 4
    R["tahap1"] = {"val_acc": best, "test_acc": accuracy(mlp, xte, yte, dev)}
    print("[T1]", R["tahap1"])
    if R["tahap1"]["test_acc"] < 0.88:
        print("PERINGATAN: test acc < 88% -> naikkan --epochs / tuning dulu sebelum lanjut.")
    torch.save(mlp.state_dict(), os.path.join(out, "mlp.pt"))

    W1, W2, W3 = (mlp.fc1.weight.cpu(), mlp.fc2.weight.cpu(), mlp.fc3.weight.cpu())
    R["orthogonality"] = {"W1": orth_stats(W1), "W2": orth_stats(W2), "W3": orth_stats(W3)}

    # 2. proyeksi balike one-hot kelas
    y1h = torch.eye(10)
    z_hat = F.relu(y1h @ W3); h1_hat = F.relu(z_hat @ W2); x_cls = torch.sigmoid(h1_hat @ W1).view(-1, 1, 28, 28)

    # 3. transpose projection dari z_MLP test image
    with torch.no_grad():
        h1_te, z_te = mlp.features(xte.to(dev))
    h1_te, z_te = h1_te.cpu(), z_te.cpu()
    x_tp = torch.sigmoid(F.relu(z_te @ W2) @ W1).view(-1, 1, 28, 28)
    R["tahap3_transpose_projection"] = recon_metrics(xte, x_tp)
    R["latent_stats"] = {"z_zero_fraction": (z_te == 0).float().mean().item(),
                         "h1_zero_fraction": (h1_te == 0).float().mean().item(),
                         "z_dead_dims": int(((z_te == 0).all(0)).sum().item())}

    # 4. frozen  latent decoding vs Vanilla AE
    with torch.no_grad():
        z_tr, z_va = mlp.features(xtr.to(dev))[1].cpu(), mlp.features(xva.to(dev))[1].cpu()
    dec = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid()).to(dev)
    od = torch.optim.Adam(dec.parameters(), lr=1e-3)
    ztr_d, xtr_d = z_tr.to(dev), xtr.flatten(1).to(dev)
    bd, bsd = float("inf"), None
    for ep in range(1, args.dec_epochs + 1):
        perm = torch.randperm(len(ztr_d), device=dev)
        for i in range(0, len(perm), 128):
            idx = perm[i:i + 128]
            loss = F.mse_loss(dec(ztr_d[idx]), xtr_d[idx])
            od.zero_grad(); loss.backward(); od.step()
        with torch.no_grad():
            v = F.mse_loss(dec(z_va.to(dev)), xva.flatten(1).to(dev)).item()
        if v < bd:
            bd, bsd = v, {k: t.clone() for k, t in dec.state_dict().items()}
    dec.load_state_dict(bsd); dec.eval()
    with torch.no_grad():
        x_fd = dec(z_te.to(dev)).cpu().view(-1, 1, 28, 28)
    R["tahap4_frozen_decoder"] = recon_metrics(xte, x_fd)

    x_ae = None
    ae = load_model("ae", args.runs, dev)
    if ae is not None:
        with torch.no_grad():
            x_ae = torch.cat([ae[0].reconstruct(xte[i:i + 1000].to(dev)).cpu() for i in range(0, len(xte), 1000)])
        R["tahap4_vanilla_ae_z64"] = recon_metrics(xte, x_ae)
    else:
        print("PERINGATAN: runs/ae belum ada -> bandingkan dengan Vanilla AE setelah train.py --model ae.")

    # 5.  sampling z
    mu, sd = z_te.mean(0), z_te.std(0)
    z_rand = mu + sd * torch.randn(64, 64, generator=torch.Generator().manual_seed(7))
    with torch.no_grad():
        x_rand = dec(z_rand.to(dev)).cpu().view(-1, 1, 28, 28)
    R["tahap5_sampling"] = {"frac_sampled_z_with_negative": (z_rand < 0).any(1).float().mean().item(),
                            "note": "z_MLP selalu >=0 (ReLU), sampel Gaussian bisa negatif -> di luar support data"}

    save_json(R, os.path.join(out, "results.json"))
    torch.save({"x": xte[:16], "y": yte[:16], "x_cls": x_cls, "x_tp": x_tp[:16], "x_fd": x_fd[:16],
                "x_ae": None if x_ae is None else x_ae[:16], "x_rand": x_rand[:16]}, os.path.join(out, "vis.pt"))
    print(R)

In [ ]:
# evaluate
RECON = ("ae", "sparse", "vae")
GENERATIVE = ("vae", "gan", "dcgan", "ddpm")
LABEL = {"ae": "Vanilla Autoencoder", "sparse": "Sparse Autoencoder", "vae": "Variational Autoencoder (VAE)",
         "gan": "GAN", "dcgan": "DCGAN", "ddpm": "DDPM"}

def fid_score(real, fake, dev, bs=100):
    from torchmetrics.image.fid import FrechetInceptionDistance
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(dev)
    for i in range(0, real.size(0), bs):
        fid.update(real[i:i + bs].to(dev).repeat(1, 3, 1, 1).clamp(0, 1), real=True)
    for i in range(0, fake.size(0), bs):
        fid.update(fake[i:i + bs].to(dev).repeat(1, 3, 1, 1).clamp(0, 1), real=False)
    return fid.compute().item()

def main_eval(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", default="runs")
    ap.add_argument("--models", nargs="+", default=ALL)
    ap.add_argument("--fid_n", type=int, default=5000, help="jumlah sampel FID (SAMA untuk semua model)")
    ap.add_argument("--skip_fid", action="store_true")
    ap.add_argument("--data_root", default="./data")
    ap.add_argument("--num_workers", type=int, default=2)
    args = ap.parse_args(argv)
    set_seed()
    dev = get_device()
    test_x = torch.cat([x for x, _ in get_loaders(128, args.data_root, args.num_workers)["test"]])
    real = test_x[:args.fid_n]
    B = 64
    res = {}

    for name in args.models:
        loaded = load_model(name, args.runs, dev)
        if loaded is None:
            print(f"[skip] {name}: belum ada checkpoint"); continue
        model, cfg, diff, ck = loaded
        log = ck["log"]
        r = {"params": count_params(model), "epochs": len(log),
             "train_time_per_epoch_s": sum(e["epoch_time"] for e in log) / len(log)}

        # metrik sparsity untuk Q1
        if name == "sparse":
          with torch.no_grad():
            z = model.encode(test_x[:1000].to(dev))

          r["frac_zero_latent"] = (z == 0).float().mean().item()
          r["mean_abs_z"] = z.abs().mean().item()
          r["dead_units"] = int((z == 0).all(0).sum().item())

        if name in RECON:
            with torch.no_grad():
                rec = torch.cat([model.reconstruct(test_x[i:i + 1000].to(dev)).cpu()
                                 for i in range(0, len(test_x), 1000)])
            r.update(recon_metrics(test_x, rec))
            if name == "sparse":
                with torch.no_grad():
                    z = torch.cat([model.encode(test_x[i:i + 1000].to(dev)).cpu() for i in range(0, len(test_x), 1000)])
                r["frac_zero_latent"] = (z == 0).float().mean().item()
                r["mean_abs_z"] = z.abs().mean().item()
                r["dead_units"] = int((z == 0).all(0).sum().item())

        xb = test_x[:B].to(dev)
        if name in ("ae", "sparse"):
            fn = lambda: model.reconstruct(xb)
        elif name == "vae":
            fn = lambda: model.decode(torch.randn(B, cfg["latent"], device=dev))
        elif name in ("gan", "dcgan"):
            fn = lambda: model(torch.randn(B, cfg["zdim"], device=dev))
        else:
            fn = lambda: diff.sample(model, B, dev)
        r["latency_ms_batch64"] = measure_latency(fn, dev, *( (0, 1) if name == "ddpm" else (5, 20) ))

        if name in GENERATIVE and not args.skip_fid:
            if name == "ddpm":
                est = r["latency_ms_batch64"] / 1000 * (args.fid_n / B) / 60
                print(f"[ddpm] estimasi waktu sampling untuk FID: ~{est:.1f} menit (turunkan --fid_n jika kelamaan)")
            t0 = time.time()
            gen = torch.Generator(device=dev).manual_seed(123)
            fake = generate(name, model, cfg, diff, args.fid_n, dev, bs=250 if name == "ddpm" else 1000,
                            gen=gen if name != "ddpm" else None)
            r["fid"] = fid_score(real, fake, dev)
            r["fid_n"] = args.fid_n
            print(f"[{name}] FID dihitung dalam {time.time() - t0:.0f}s")
        res[name] = r
        print(name, {k: (round(v, 5) if isinstance(v, float) else v) for k, v in r.items()})

    save_json(res, os.path.join(args.runs, "results.json"))
    f = lambda r, k, d=4: f"{r[k]:.{d}f}" if k in r else "N/A"
    lines = ["| Model | Params | Train time/epoch (s) | Latency (ms/batch 64) | Test MSE | Test PSNR (dB) | Test SSIM | FID |",
             "|---|---|---|---|---|---|---|---|"]
    for n, r in res.items():
        lines.append(f"| {LABEL[n]} | {r['params']:,} | {r['train_time_per_epoch_s']:.1f} | {r['latency_ms_batch64']:.2f} | "
                     f"{f(r,'mse',5)} | {f(r,'psnr',2)} | {f(r,'ssim',4)} | {f(r,'fid',2)} |")
    with open(os.path.join(args.runs, "results_table.md"), "w") as fh:
        fh.write("\n".join(lines) + "\n")
    print("\n".join(lines))

In [ ]:
# figures
DPI = 220


def strip(t, nrow=None):
    """tensor (N,1,28,28) di [0,1] -> array gambar (H,W) untuk imshow."""
    return make_grid(t.clamp(0, 1), nrow=nrow or t.size(0), padding=1, pad_value=1.0)[0].numpy()


def rows_figure(rows, path, title=None, cell=0.9):
    n = max(r[1].size(0) for r in rows)
    fig, axs = plt.subplots(len(rows), 1, figsize=(max(6, n * cell), len(rows) * cell * 1.15 + 0.6))
    axs = [axs] if len(rows) == 1 else axs
    for ax, (label, t) in zip(axs, rows):
        ax.imshow(strip(t), cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=9)
        for s in ax.spines.values():
            s.set_visible(False)
    if title:
        fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(path, dpi=DPI, bbox_inches="tight"); plt.close(fig)
    print("saved", path)


def main_fig(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", default="runs")
    ap.add_argument("--out", default="figures")
    ap.add_argument("--data_root", default="./data")
    ap.add_argument("--class_a", type=int, default=1, help="kelas citra A untuk interpolasi (1=Trouser)")
    ap.add_argument("--class_b", type=int, default=9, help="kelas citra B untuk interpolasi (9=Ankle boot)")
    ap.add_argument("--num_workers", type=int, default=2)
    args = ap.parse_args(argv)
    set_seed()
    dev = get_device()
    os.makedirs(args.out, exist_ok=True)
    M = {n: load_model(n, args.runs, dev) for n in ("ae", "sparse", "vae", "gan", "dcgan", "ddpm")}
    xs, ys = zip(*[(x, y) for x, y in get_loaders(128, args.data_root, args.num_workers)["test"]])
    x_te, y_te = torch.cat(xs), torch.cat(ys)

    # ---------- Fig 1: grid rekonstruksi 8 sampel
    if all(M[n] for n in ("ae", "sparse", "vae")):
        x8 = x_te[:8]
        rows = [("Asli", x8)]
        for n, lab in (("ae", "Vanilla AE"), ("sparse", "Sparse AE"), ("vae", "VAE")):
            with torch.no_grad():
                rows.append((lab, M[n][0].reconstruct(x8.to(dev)).cpu()))
        rows_figure(rows, f"{args.out}/fig1_reconstruction.png", "Figur 1: Rekonstruksi (Test Set)")

    # ---------- Fig 2: sampling acak 4x4 dari 4 model
    panels = []
    for n, lab in (("vae", "A: VAE"), ("gan", "B: Vanilla GAN"), ("dcgan", "C: DCGAN"), ("ddpm", "D: DDPM")):
        if M[n] is None:
            continue
        model, cfg, diff, _ = M[n]
        g = torch.Generator(device=dev).manual_seed(1)
        with torch.no_grad():
            if n == "vae":
                s = model.decode(torch.randn(16, cfg["latent"], device=dev, generator=g))
            elif n == "ddpm":
                s = diff.sample(model, 16, dev) * 0.5 + 0.5
            else:
                s = model(torch.randn(16, cfg["zdim"], device=dev, generator=g)) * 0.5 + 0.5
        panels.append((lab, s.cpu()))
    if panels:
        fig, axs = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.5))
        axs = [axs] if len(panels) == 1 else axs
        for ax, (lab, s) in zip(axs, panels):
            ax.imshow(strip(s, 4), cmap="gray", vmin=0, vmax=1); ax.set_title(lab, fontsize=10); ax.axis("off")
        fig.suptitle("Figur 2: Random generative sampling 4x4", fontsize=11)
        fig.tight_layout(); fig.savefig(f"{args.out}/fig2_random_sampling.png", dpi=DPI, bbox_inches="tight"); plt.close(fig)
        print("saved fig2")

    # ---------- Fig 3: bukti batasan MLP (butuh runs/mlp/vis.pt)
    vp = os.path.join(args.runs, "mlp", "vis.pt")
    if os.path.exists(vp):
        v = torch.load(vp, weights_only=False)
        rows = [("(1) Asli", v["x"][:10]),
                ("(2) W^T dari one-hot\nkelas (Tahap 2)", v["x_cls"]),
                ("(2b) W^T dari z_MLP\n(Tahap 3)", v["x_tp"][:10]),
                ("(3) Frozen decoder\n(z_MLP -> x)", v["x_fd"][:10])]
        if v["x_ae"] is not None:
            rows.append(("(3b) Vanilla AE (z_AE)", v["x_ae"][:10]))
        rows.append(("(4) Sampling\nN(mu_z, sigma_z^2)", v["x_rand"][:10]))
        rows_figure(rows, f"{args.out}/fig3_mlp_limitation.png", "Figur 3: Batasan MLP Classifier untuk image generation")
    else:
        print("[skip fig3] jalankan mlp_experiment.py dulu")

    # ---------- Fig 4: interpolasi linier z(a) = (1-a) zA + a zB
    alphas = torch.tensor([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ia, ib = (y_te == args.class_a).nonzero()[0].item(), (y_te == args.class_b).nonzero()[0].item()
    xa, xb = x_te[ia:ia + 1].to(dev), x_te[ib:ib + 1].to(dev)
    rows = []
    with torch.no_grad():
        for n, lab in (("ae", "Vanilla AE"), ("sparse", "Sparse AE"), ("vae", "VAE")):
            if M[n] is None:
                continue
            m = M[n][0]
            za, zb = m.encode_mean(xa), m.encode_mean(xb)
            z = (1 - alphas[:, None].to(dev)) * za + alphas[:, None].to(dev) * zb
            rows.append((lab, m.decode(z).cpu()))
        for n, lab in (("gan", "Vanilla GAN"), ("dcgan", "DCGAN")):
            if M[n] is None:
                continue
            g = torch.Generator(device=dev).manual_seed(3)
            za, zb = (torch.randn(1, M[n][1]["zdim"], device=dev, generator=g) for _ in range(2))
            z = (1 - alphas[:, None].to(dev)) * za + alphas[:, None].to(dev) * zb
            rows.append((lab, (M[n][0](z) * 0.5 + 0.5).cpu()))
    if rows:
        rows_figure(rows, f"{args.out}/fig4_latent_interpolation.png",
                    "Figur 4: Interpolasi linier, alpha = 0, 0.2, ..., 1.0")

    # ---------- Fig 5: kurva loss + trajektori reverse diffusion
    logs = {n: load_json(os.path.join(args.runs, n, "log.json")) for n in M if os.path.exists(os.path.join(args.runs, n, "log.json"))}
    fig = plt.figure(figsize=(16, 7.5))
    gs = fig.add_gridspec(2, 4, height_ratios=[1, 1.1])
    ax = fig.add_subplot(gs[0, 0])
    for n, lab in (("ae", "Vanilla AE"), ("sparse", "Sparse AE"), ("vae", "VAE")):
        if n in logs:
            ax.plot([e["epoch"] for e in logs[n]], [e["val_mse"] for e in logs[n]], label=lab)
    ax.set_title("Val MSE rekonstruksi"); ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.legend()
    for k, n, lab in ((1, "gan", "Vanilla GAN"), (2, "dcgan", "DCGAN")):
        ax = fig.add_subplot(gs[0, k])
        if n in logs:
            ax.plot([e["epoch"] for e in logs[n]], [e["loss_g"] for e in logs[n]], label="Generator")
            ax.plot([e["epoch"] for e in logs[n]], [e["loss_d"] for e in logs[n]], label="Discriminator")
        ax.set_title(f"{lab}: loss G & D"); ax.set_xlabel("epoch"); ax.legend()
    ax = fig.add_subplot(gs[0, 3])
    if "ddpm" in logs:
        ax.plot([e["epoch"] for e in logs["ddpm"]], [e["train_loss"] for e in logs["ddpm"]], label="train")
        ax.plot([e["epoch"] for e in logs["ddpm"]], [e["val_loss"] for e in logs["ddpm"]], label="val")
    ax.set_title("DDPM: loss prediksi noise"); ax.set_xlabel("epoch"); ax.legend()
    ax = fig.add_subplot(gs[1, :])
    if M["ddpm"] is not None:
        model, cfg, diff, _ = M["ddpm"]
        T = cfg["T"]
        ts = [T - 1, int(.9 * T), int(.8 * T), int(.6 * T), int(.4 * T), int(.2 * T), int(.1 * T), int(.05 * T)]
        torch.manual_seed(5)
        _, traj = diff.sample(model, 3, dev, save_ts=set(ts))
        cols = [(t, x.cpu() * 0.5 + 0.5) for t, x in traj]
        big = torch.cat([torch.cat([c[1][i:i + 1] for c in cols]) for i in range(3)])  # 3 baris x len(cols)
        ax.imshow(strip(big, len(cols)), cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([(i + 0.5) * 29 for i in range(len(cols))])
        ax.set_xticklabels([f"t={t}" if t >= 0 else "x0" for t, _ in cols]); ax.set_yticks([])
        ax.set_title("Reverse diffusion x_T -> x_0 (3 sampel)")
    fig.suptitle("Figur 5: Training loss curves & trajektori denoising DDPM", fontsize=12)
    fig.tight_layout(); fig.savefig(f"{args.out}/fig5_losses_and_ddpm.png", dpi=DPI, bbox_inches="tight"); plt.close(fig)
    print("saved fig5")

In [ ]:
main_mlp(["--runs", RUNS])

[T1] epoch 1 val_acc=0.8421
[T1] epoch 2 val_acc=0.8599
[T1] epoch 3 val_acc=0.8567
[T1] epoch 4 val_acc=0.8743
[T1] epoch 5 val_acc=0.8704
[T1] epoch 6 val_acc=0.8741
[T1] epoch 7 val_acc=0.8767
[T1] epoch 8 val_acc=0.8797
[T1] epoch 9 val_acc=0.8836
[T1] epoch 10 val_acc=0.8861
[T1] epoch 11 val_acc=0.8884
[T1] epoch 12 val_acc=0.8846
[T1] epoch 13 val_acc=0.8894
[T1] epoch 14 val_acc=0.8917
[T1] epoch 15 val_acc=0.8911
[T1] epoch 16 val_acc=0.8890
[T1] epoch 17 val_acc=0.8951
[T1] epoch 18 val_acc=0.8931
[T1] epoch 19 val_acc=0.8833
[T1] epoch 20 val_acc=0.8933
[T1] {'val_acc': 0.8951428571428571, 'test_acc': 0.8875714285714286}
{'tahap1': {'val_acc': 0.8951428571428571, 'test_acc': 0.8875714285714286}, 'orthogonality': {'W1': {'shape': [256, 784], 'fro_WtW_minus_I': 396.93951416015625, 'fro_WWt_minus_I': 396.2778625488281, 'fro_WWt_minus_I_normalized': 24.767366409301758, 'sv_max': 12.87828254699707, 'sv_min': 0.3679221272468567, 'cond_number': 35.00273895263672}, 'W2': {'shape': [

In [ ]:
main_eval(["--runs", RUNS, "--fid_n", "5000"])   # DDPM FID lama; turunkan --fid_n jika perlu

ae {'params': 435536, 'epochs': 30, 'train_time_per_epoch_s': 7.15581, 'mse': 0.00651, 'psnr': 22.98918, 'ssim': 0.78894, 'latency_ms_batch64': 0.16807}
sparse {'params': 1067024, 'epochs': 30, 'train_time_per_epoch_s': 7.86351, 'frac_zero_latent': 0.37707, 'mean_abs_z': 0.32935, 'dead_units': 66, 'mse': 0.00444, 'psnr': 24.68252, 'ssim': 0.84495, 'latency_ms_batch64': 0.25401}
[vae] FID dihitung dalam 53s
vae {'params': 451984, 'epochs': 30, 'train_time_per_epoch_s': 7.18528, 'mse': 0.01322, 'psnr': 19.63938, 'ssim': 0.64794, 'latency_ms_batch64': 0.11705, 'fid': 85.84397, 'fid_n': 5000}
[gan] FID dihitung dalam 43s
gan {'params': 1477136, 'epochs': 50, 'train_time_per_epoch_s': 9.09947, 'latency_ms_batch64': 0.29605, 'fid': 103.01206, 'fid_n': 5000}
[dcgan] FID dihitung dalam 45s
dcgan {'params': 759681, 'epochs': 30, 'train_time_per_epoch_s': 8.99099, 'latency_ms_batch64': 0.95378, 'fid': 22.4842, 'fid_n': 5000}
[ddpm] estimasi waktu sampling untuk FID: ~20.9 menit (turunkan --fid_n

In [ ]:
main_fig(["--runs", RUNS, "--out", f"{RUNS}/figures"])

saved /content/drive/MyDrive/AssignmentDLdrive/figures/fig1_reconstruction.png
saved fig2
saved /content/drive/MyDrive/AssignmentDLdrive/figures/fig3_mlp_limitation.png
saved /content/drive/MyDrive/AssignmentDLdrive/figures/fig4_latent_interpolation.png
saved fig5
